In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

DEFAULTS = {
    "catalog_name": "catalog_smartdata",

    "raw_schema": "raw",
    "bronze_schema": "bronze",
    "silver_schema": "silver",
    "gold_schema": "gold",

    "storage_account": "adlsjrdata",

    "raw_container": "raw",
    "bronze_container": "bronze",
    "silver_container": "silver",
    "gold_container": "gold",

    "storage_credential_name": "credential",

    "external_location_raw": "exlt-raw",
    "external_location_bronze": "exlt-bronze",
    "external_location_silver": "exlt-silver",
    "external_location_gold": "exlt-gold",

    "access_connector_or_mi_name": "ac-jr-access",
}

def _safe_get_widget(name: str, default_value: str) -> str:
    try:
        dbutils.widgets.text(name, default_value)
        return dbutils.widgets.get(name)
    except Exception:
        return default_value

catalog_name = _safe_get_widget("catalog_name", DEFAULTS["catalog_name"]).strip()

raw_schema = _safe_get_widget("raw_schema", DEFAULTS["raw_schema"]).strip()
bronze_schema = _safe_get_widget("bronze_schema", DEFAULTS["bronze_schema"]).strip()
silver_schema = _safe_get_widget("silver_schema", DEFAULTS["silver_schema"]).strip()
gold_schema = _safe_get_widget("gold_schema", DEFAULTS["gold_schema"]).strip()

storage_account = _safe_get_widget("storage_account", DEFAULTS["storage_account"]).strip()

raw_container = _safe_get_widget("raw_container", DEFAULTS["raw_container"]).strip()
bronze_container = _safe_get_widget("bronze_container", DEFAULTS["bronze_container"]).strip()
silver_container = _safe_get_widget("silver_container", DEFAULTS["silver_container"]).strip()
gold_container = _safe_get_widget("gold_container", DEFAULTS["gold_container"]).strip()

storage_credential_name = _safe_get_widget("storage_credential_name", DEFAULTS["storage_credential_name"]).strip()

external_location_raw = _safe_get_widget("external_location_raw", DEFAULTS["external_location_raw"]).strip()
external_location_bronze = _safe_get_widget("external_location_bronze", DEFAULTS["external_location_bronze"]).strip()
external_location_silver = _safe_get_widget("external_location_silver", DEFAULTS["external_location_silver"]).strip()
external_location_gold = _safe_get_widget("external_location_gold", DEFAULTS["external_location_gold"]).strip()

access_connector_or_mi_name = _safe_get_widget(
    "access_connector_or_mi_name", DEFAULTS["access_connector_or_mi_name"]
).strip()


spark.sql(f"USE CATALOG {catalog_name}")

raw_uri = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net"
bronze_uri = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"
silver_uri = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"
gold_uri = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net"

bronze_catalog = f"{catalog_name}.{bronze_schema}"
silver_catalog = f"{catalog_name}.{silver_schema}"
gold_catalog = f"{catalog_name}.{gold_schema}"

source_files = {
    "automobile": f"{raw_uri}/Automobile_data.csv"
}

bronze_tables = {
    "automobile": f"{bronze_catalog}.automobile"
}

silver_tables = {
    "autos_clean": f"{silver_catalog}.autos_clean",
    "engine_specs": f"{silver_catalog}.engine_specs",
    "vehicle_specs": f"{silver_catalog}.vehicle_specs",
    "autos_enriched": f"{silver_catalog}.autos_enriched",
    "autos_conformed": f"{silver_catalog}.autos_conformed",
}

gold_tables = {
    "dim_make": f"{gold_catalog}.dim_make",
    "dim_body_style": f"{gold_catalog}.dim_body_style",
    "dim_engine": f"{gold_catalog}.dim_engine",
    "dim_fuel": f"{gold_catalog}.dim_fuel",
    "dim_drive": f"{gold_catalog}.dim_drive",
    "fact_autos": f"{gold_catalog}.fact_autos",
}

def save_delta_table(df, full_table_name: str):
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

def print_runtime_config():
    print("CONFIG FINAL:")
    print(f"catalog={catalog_name}")
    print(f"raw_uri={raw_uri}")
    print(f"bronze={bronze_catalog}")
    print(f"silver={silver_catalog}")
    print(f"gold={gold_catalog}")
    print(f"access_connector={access_connector_or_mi_name}")

print_runtime_config()